The GLMMs run to produce the results reported in Glitz et al (2026), eLife. Note that the models are in the order they are reported in in the paper.

In [1]:
library(lme4)
library(ggplot2)
library(dplyr)

Loading required package: Matrix



**Data loading and prep**

In [13]:
#load dataframes
train_df <- read.csv('/Users/leonieglitz/Desktop/Grids_paper/training_acc_longform.csv')
nav_df <- read.csv('/Users/leonieglitz/Desktop/Grids_paper/twoStep_acc_longform.csv')
prox_df <- read.csv('/Users/leonieglitz/Desktop/Grids_paper/prox_acc_longform.csv')
oneStep_df <- read.csv('/Users/leonieglitz/Desktop/Grids_paper/oneStep_acc_longform.csv')


In [6]:
#convert categorical vars to factors 
train_df$condition <- factor(train_df$condition)
train_df$group <- factor(train_df$group)
train_df$shape <- factor(train_df$shape)
train_df$trained_first <- factor(train_df$trained_first)

nav_df$condition <- factor(nav_df$condition)
nav_df$group <- factor(nav_df$group)
nav_df$shape <- factor(nav_df$shape)
nav_df$blockNo <- factor(nav_df$blockNo)

prox_df$condition <- factor(prox_df$condition)
prox_df$group <- factor(prox_df$group)
prox_df$shape <- factor(prox_df$shape)
prox_df$closer_dist <- factor(prox_df$closer_dist)

oneStep_df$condition <- factor(oneStep_df$condition)
oneStep_df$group <- factor(oneStep_df$group)
oneStep_df$shape <- factor(oneStep_df$shape)

In [7]:
#centre one-step acc as a predictor
nav_df$oneStepAcc_c <- nav_df$oneStepAcc - mean(
  nav_df$oneStepAcc,
  na.rm = TRUE
)

prox_df$oneStepAcc_c <- prox_df$oneStepAcc - mean(
  prox_df$oneStepAcc,
  na.rm = TRUE
)

**training accuracy**: generalised mixed effects models with a logit linking function

In [23]:
#overall training accuracy
model_train <- glmer(
  correct_train_choice ~ condition* group + shape +
    (1 + condition | subID),
  data = train_df,
  family = binomial(link = "logit"),
  control = glmerControl(
    optimizer = "Nelder_Mead",
    optCtrl = list(maxfun = 100000)
  )
)

summary(model_train)
confint(model_train, parm = "beta_", method = "Wald")

Generalized linear mixed model fit by maximum likelihood (Laplace
  Approximation) [glmerMod]
 Family: binomial  ( logit )
Formula: correct_train_choice ~ condition * group + shape + (1 + condition |  
    subID)
   Data: train_df
Control: 
glmerControl(optimizer = "Nelder_Mead", optCtrl = list(maxfun = 1e+05))

      AIC       BIC    logLik -2*log(L)  df.resid 
 584024.5  584116.8 -292004.3  584008.5    757930 

Scaled residuals: 
     Min       1Q   Median       3Q      Max 
-10.3209   0.2072   0.2939   0.4513   1.0993 

Random effects:
 Groups Name        Variance Std.Dev. Corr  
 subID  (Intercept) 1.5354   1.2391         
        condition   0.4887   0.6991   -0.75 
Number of obs: 757938, groups:  subID, 482

Fixed effects:
                Estimate Std. Error z value Pr(>|z|)    
(Intercept)      2.05243    0.31588   6.497 8.17e-11 ***
condition        0.64280    0.12902   4.982 6.29e-07 ***
group            0.02245    0.20086   0.112    0.911    
shape           -0.66677    0.124

,2.5 %,97.5 %
(Intercept),1.4333167,2.6715448
condition,0.3899187,0.8956812
group,-0.3712210,0.4161277
shape,-0.9101994,-0.4233333
condition:group,-0.1180197,0.3303105


In [24]:
#day 1/2 training accuracy
model_train_initial <- glmer(
  correct_train_choice ~ condition * group + shape +
    (1 + condition | subID),
  data = train_df,
  subset = train_type == 1,
  family = binomial(link = "logit"),
  control = glmerControl(
    optimizer = "Nelder_Mead",
    optCtrl = list(maxfun = 100000)
  )
)

summary(model_train_initial)
confint(model_train_initial, parm = "beta_", method = "Wald")

#refresher training accuracy
model_train_refresher <- glmer(
  correct_train_choice ~ condition * group + shape +
    (1 + condition | subID),
  data = train_df,
  subset = train_type == 2,
  family = binomial(link = "logit"),
  control = glmerControl(
    optimizer = "Nelder_Mead",
    optCtrl = list(maxfun = 100000)
  )
)

summary(model_train_refresher)
confint(model_train_refresher, parm = "beta_", method = "Wald")

Generalized linear mixed model fit by maximum likelihood (Laplace
  Approximation) [glmerMod]
 Family: binomial  ( logit )
Formula: correct_train_choice ~ condition * trained_first + group + shape +  
    (1 + condition | subID)
   Data: train_df
Control: 
glmerControl(optimizer = "Nelder_Mead", optCtrl = list(maxfun = 1e+05))
 Subset: train_type == 1

      AIC       BIC    logLik -2*log(L)  df.resid 
 414166.9  414267.1 -207074.5  414148.9    505155 

Scaled residuals: 
    Min      1Q  Median      3Q     Max 
-8.8609  0.2080  0.3058  0.4908  1.0422 

Random effects:
 Groups Name        Variance Std.Dev. Corr  
 subID  (Intercept) 1.4045   1.1851         
        condition   0.5645   0.7513   -0.76 
Number of obs: 505164, groups:  subID, 482

Fixed effects:
                        Estimate Std. Error z value Pr(>|z|)    
(Intercept)              1.45701    0.30031   4.852 1.22e-06 ***
condition                1.00410    0.11467   8.756  < 2e-16 ***
trained_first            0.15170   

,2.5 %,97.5 %
(Intercept),0.86841799,2.04560808
condition,0.77934252,1.22885608
trained_first,-0.06722169,0.37061997
group,-0.10602189,0.37475630
shape,-0.87518282,-0.41493823
condition:trained_first,-0.26671562,0.01236222


**Navigation test: one-step retention accuracy**

In [25]:
model_oneStep <- glmer(
  goalReached ~ condition * group + shape +
    (1 + condition | subID),
  data = oneStep_df,
  family = binomial(link = "logit"),
  control = glmerControl(
    optimizer = "Nelder_Mead",
    optCtrl = list(maxfun = 100000)
  )
)

summary(model_oneStep)
confint(model_oneStep, parm = "beta_", method = "Wald")

Generalized linear mixed model fit by maximum likelihood (Laplace
  Approximation) [glmerMod]
 Family: binomial  ( logit )
Formula: goalReached ~ condition * group + shape + (1 + condition | subID)
   Data: oneStep_df
Control: 
glmerControl(optimizer = "Nelder_Mead", optCtrl = list(maxfun = 1e+05))

      AIC       BIC    logLik -2*log(L)  df.resid 
  11712.0   11769.4   -5848.0   11696.0      9631 

Scaled residuals: 
    Min      1Q  Median      3Q     Max 
-3.6142 -0.8013  0.3940  0.7118  1.8402 

Random effects:
 Groups Name        Variance Std.Dev. Corr  
 subID  (Intercept) 3.3144   1.8206         
        condition   0.5001   0.7072   -0.90 
Number of obs: 9639, groups:  subID, 482

Fixed effects:
                Estimate Std. Error z value Pr(>|z|)    
(Intercept)      3.14774    0.54060   5.823 5.79e-09 ***
condition       -0.60997    0.22878  -2.666  0.00767 ** 
group            0.53492    0.40470   1.322  0.18624    
shape           -1.11389    0.16501  -6.750 1.48e-11 ***
c

,2.5 %,97.5 %
(Intercept),2.0881805,4.2073004
condition,-1.0583680,-0.1615652
group,-0.2582657,1.3281114
shape,-1.4373093,-0.7904654
condition:group,-0.4673596,0.3327245


**Navigation test: two-step accuracy**

In [93]:
#overall two-step accuracy
model_twoStep <- glmer(
  goalReached ~ condition * group + shape + oneStepAcc_c +
    (1 + condition | subID),
  data = nav_df,
  family = binomial(link = "logit"),
  control = glmerControl(
    optimizer = "Nelder_Mead",
    optCtrl = list(maxfun = 100000)
  )
)

summary(model_twoStep)
confint(model_twoStep, parm = "beta_", method = "Wald")

Generalized linear mixed model fit by maximum likelihood (Laplace
  Approximation) [glmerMod]
 Family: binomial  ( logit )
Formula: goalReached ~ condition * group + shape + oneStepAcc_c + (1 +  
    condition | subID)
   Data: nav_df
Control: 
glmerControl(optimizer = "Nelder_Mead", optCtrl = list(maxfun = 1e+05))

      AIC       BIC    logLik -2*log(L)  df.resid 
  18407.5   18477.8   -9194.7   18389.5     18188 

Scaled residuals: 
    Min      1Q  Median      3Q     Max 
-6.0216 -0.6217  0.2336  0.5839  4.0508 

Random effects:
 Groups Name        Variance Std.Dev. Corr  
 subID  (Intercept) 0.9673   0.9835         
        condition2  0.8707   0.9331   -0.59 
Number of obs: 18197, groups:  subID, 182

Fixed effects:
                  Estimate Std. Error z value Pr(>|z|)    
(Intercept)        0.81272    0.12710   6.394 1.61e-10 ***
condition2        -0.40515    0.11155  -3.632 0.000281 ***
group2             0.11664    0.15955   0.731 0.464748    
shape2             0.04287    0.

,2.5 %,97.5 %
(Intercept),0.5636021,1.0618440
condition2,-0.6237794,-0.1865127
group2,-0.1960771,0.4293599
shape2,-0.2188233,0.3045666
oneStepAcc_c,5.4225445,6.6074925
condition2:group2,-0.4607804,0.1727069


In [94]:
#diagonal two-step accuracy
model_diag <- glmer(
  goalReached ~ condition * group + shape + oneStepAcc_c +
    (1 + condition | subID),
  data = nav_df,
  subset = twoStepType == 1,
  family = binomial(link = "logit"),
  control = glmerControl(
    optimizer = "Nelder_Mead",
    optCtrl = list(maxfun = 100000)
  )
)

summary(model_diag)
confint(model_diag, parm = "beta_", method = "Wald")

Generalized linear mixed model fit by maximum likelihood (Laplace
  Approximation) [glmerMod]
 Family: binomial  ( logit )
Formula: goalReached ~ condition * group + shape + oneStepAcc_c + (1 +  
    condition | subID)
   Data: nav_df
Control: 
glmerControl(optimizer = "Nelder_Mead", optCtrl = list(maxfun = 1e+05))
 Subset: twoStepType == 1

      AIC       BIC    logLik -2*log(L)  df.resid 
   9259.6    9324.0   -4620.8    9241.6      9454 

Scaled residuals: 
    Min      1Q  Median      3Q     Max 
-5.3349 -0.5505  0.2159  0.5104  4.9068 

Random effects:
 Groups Name        Variance Std.Dev. Corr  
 subID  (Intercept) 1.367    1.169          
        condition2  1.394    1.181    -0.61 
Number of obs: 9463, groups:  subID, 182

Fixed effects:
                  Estimate Std. Error z value Pr(>|z|)    
(Intercept)         0.8592     0.1555   5.525  3.3e-08 ***
condition2         -0.5635     0.1462  -3.853 0.000117 ***
group2              0.1799     0.1956   0.919 0.357853    
shape2 

,2.5 %,97.5 %
(Intercept),0.5543627,1.1639960
condition2,-0.8501408,-0.2768519
group2,-0.2035426,0.5632733
shape2,-0.7406477,-0.1102734
oneStepAcc_c,5.6815995,7.1363139
condition2:group2,-0.6704784,0.1606211


In [95]:
#straight two-step accuracy
model_straight <- glmer(
  goalReached ~ condition * group + shape + oneStepAcc_c +
    (1 + condition | subID),
  data = nav_df,
  subset = twoStepType == 0,
  family = binomial(link = "logit"),
  control = glmerControl(
    optimizer = "Nelder_Mead",
    optCtrl = list(maxfun = 100000)
  )
)

summary(model_straight)
confint(model_straight, parm = "beta_", method = "Wald")

Generalized linear mixed model fit by maximum likelihood (Laplace
  Approximation) [glmerMod]
 Family: binomial  ( logit )
Formula: goalReached ~ condition * group + shape + oneStepAcc_c + (1 +  
    condition | subID)
   Data: nav_df
Control: 
glmerControl(optimizer = "Nelder_Mead", optCtrl = list(maxfun = 1e+05))
 Subset: twoStepType == 0

      AIC       BIC    logLik -2*log(L)  df.resid 
   8759.7    8823.4   -4370.8    8741.7      8725 

Scaled residuals: 
    Min      1Q  Median      3Q     Max 
-4.9390 -0.6548  0.2416  0.5644  4.1201 

Random effects:
 Groups Name        Variance Std.Dev. Corr  
 subID  (Intercept) 1.0757   1.0372         
        condition2  0.8984   0.9478   -0.54 
Number of obs: 8734, groups:  subID, 182

Fixed effects:
                  Estimate Std. Error z value Pr(>|z|)    
(Intercept)        0.73293    0.14485   5.060 4.19e-07 ***
condition2        -0.23918    0.12810  -1.867  0.06188 .  
group2             0.09557    0.17868   0.535  0.59275    
shape2 

,2.5 %,97.5 %
(Intercept),0.4490313,1.01682374
condition2,-0.4902394,0.01188484
group2,-0.2546305,0.44576344
shape2,0.2051937,0.81986729
oneStepAcc_c,5.6289132,7.05838300
condition2:group2,-0.4857066,0.23641763


In [96]:
#interaction two step type 
model_interaction <- glmer(
  goalReached ~ condition * group * twoStepType + shape + oneStepAcc_c +
    (1 + condition | subID),
  data = nav_df,
  family = binomial(link = "logit"),
  control = glmerControl(
    optimizer = "Nelder_Mead",
    optCtrl = list(maxfun = 100000)
  )
)

summary(model_interaction)
confint(model_interaction, parm = "beta_", method = "Wald")

Generalized linear mixed model fit by maximum likelihood (Laplace
  Approximation) [glmerMod]
 Family: binomial  ( logit )
Formula: 
goalReached ~ condition * group * twoStepType + shape + oneStepAcc_c +  
    (1 + condition | subID)
   Data: nav_df
Control: 
glmerControl(optimizer = "Nelder_Mead", optCtrl = list(maxfun = 1e+05))

      AIC       BIC    logLik -2*log(L)  df.resid 
  18194.4   18295.9   -9084.2   18168.4     18184 

Scaled residuals: 
    Min      1Q  Median      3Q     Max 
-6.0210 -0.6066  0.2238  0.5736  4.4208 

Random effects:
 Groups Name        Variance Std.Dev. Corr  
 subID  (Intercept) 0.9792   0.9895         
        condition2  0.8897   0.9433   -0.58 
Number of obs: 18197, groups:  subID, 182

Fixed effects:
                              Estimate Std. Error z value Pr(>|z|)    
(Intercept)                    1.01431    0.13436   7.549 4.39e-14 ***
condition2                    -0.24998    0.12517  -1.997  0.04582 *  
group2                         0.05154  

,2.5 %,97.5 %
(Intercept),0.7509640,1.277664901
condition2,-0.4953185,-0.004645101
group2,-0.2830387,0.386111298
twoStepType,-0.5241620,-0.228800467
shape2,-0.2196102,0.311237254
oneStepAcc_c,5.5155110,6.717571610
condition2:group2,-0.4071309,0.305060330
condition2:twoStepType,-0.4999182,-0.095970614
group2:twoStepType,-0.0948378,0.329311297
condition2:group2:twoStepType,-0.4613051,0.124620001


In [64]:
#do participants learn during test
model_learning <- glmer(
  goalReached ~ condition * group * blockNo + shape + oneStepAcc_c +
    (1 + condition | subID),
  data = nav_df,
  family = binomial(link = "logit"),
  control = glmerControl(
    optimizer = "Nelder_Mead",
    optCtrl = list(maxfun = 100000)
  )
)

summary(model_learning)
confint(model_learning, parm = "beta_", method = "Wald")

Generalized linear mixed model fit by maximum likelihood (Laplace
  Approximation) [glmerMod]
 Family: binomial  ( logit )
Formula: goalReached ~ condition * group * blockNo + shape + oneStepAcc_c +  
    (1 + condition | subID)
   Data: nav_df
Control: 
glmerControl(optimizer = "Nelder_Mead", optCtrl = list(maxfun = 1e+05))

      AIC       BIC    logLik -2*log(L)  df.resid 
  18362.7   18464.3   -9168.4   18336.7     18184 

Scaled residuals: 
    Min      1Q  Median      3Q     Max 
-6.2674 -0.6218  0.2302  0.5874  4.2440 

Random effects:
 Groups Name        Variance Std.Dev. Corr  
 subID  (Intercept) 0.9731   0.9865         
        condition2  0.8764   0.9362   -0.59 
Number of obs: 18197, groups:  subID, 182

Fixed effects:
                           Estimate Std. Error z value Pr(>|z|)    
(Intercept)                 0.66403    0.13258   5.009 5.48e-07 ***
condition2                 -0.44033    0.12269  -3.589 0.000332 ***
group2                      0.17973    0.16838   1.067

,2.5 %,97.5 %
(Intercept),0.4041789,0.92387737
condition2,-0.6807940,-0.19987450
group2,-0.1502933,0.50974606
blockNo2,0.1563450,0.45039259
shape2,-0.2197697,0.30506186
oneStepAcc_c,5.4422426,6.63055485
condition2:group2,-0.4373063,0.25906208
condition2:blockNo2,-0.1322202,0.26821729
group2:blockNo2,-0.3411791,0.08115186
condition2:group2:blockNo2,-0.3989674,0.18041407


**Proximity judgements accuracy**

In [36]:
model_overall_prox <- glmer(
  correct_prox_choice ~ condition * group + shape + oneStepAcc_c +
    (1 + condition | subID),
  data = prox_df,
  family = binomial(link = "logit"),
  control = glmerControl(
    optimizer = "Nelder_Mead",
    optCtrl = list(maxfun = 100000)
  )
)

summary(model_overall_prox)
confint(model_overall_prox, parm = "beta_", method = "Wald")

Generalized linear mixed model fit by maximum likelihood (Laplace
  Approximation) [glmerMod]
 Family: binomial  ( logit )
Formula: correct_prox_choice ~ condition * group + shape + oneStepAcc_c +  
    (1 + condition | subID)
   Data: prox_df
Control: 
glmerControl(optimizer = "Nelder_Mead", optCtrl = list(maxfun = 1e+05))

      AIC       BIC    logLik -2*log(L)  df.resid 
  37210.0   37284.6  -18596.0   37192.0     29111 

Scaled residuals: 
    Min      1Q  Median      3Q     Max 
-2.3661 -1.1761  0.6301  0.7583  1.0981 

Random effects:
 Groups Name        Variance Std.Dev. Corr  
 subID  (Intercept) 0.07987  0.2826         
        condition2  0.08208  0.2865   -0.61 
Number of obs: 29120, groups:  subID, 182

Fixed effects:
                   Estimate Std. Error z value Pr(>|z|)    
(Intercept)        0.666434   0.043380  15.363  < 2e-16 ***
condition2        -0.235508   0.045144  -5.217 1.82e-07 ***
group2            -0.005541   0.055297  -0.100   0.9202    
shape2             

,2.5 %,97.5 %
(Intercept),0.58141116,0.7514561
condition2,-0.32398780,-0.1470283
group2,-0.11392036,0.1028387
shape2,0.03781092,0.2114619
oneStepAcc_c,1.02698319,1.4132591
condition2:group2,-0.05847427,0.1984654


In [45]:
#N.B.: this gives a convergence warning. However, I have checked and when using other solvers
#      such as bobyqa you do not get a convergence warning and very similar parameter estimates
#      for consistency's sake, I am reporting the results below as I am using Nelder_Mead for all
#      other models as well. 

model_2v1_prox <- glmer(
  correct_prox_choice ~ condition * group + shape + oneStepAcc_c +
    (1 + condition | subID),
  data = prox_df,
  subset = closer_dist==2,
  family = binomial(link = "logit"),
  control = glmerControl(
    optimizer = "Nelder_Mead",
    optCtrl = list(maxfun = 100000)
  )
)

summary(model_2v1_prox)
confint(model_2v1_prox, parm = "beta_", method = "Wald")

Warning message in checkConv(attr(opt, "derivs"), opt$par, ctrl = control$checkConv, :
“Model failed to converge with max|grad| = 0.0146657 (tol = 0.002, component 1)
  See ?lme4::convergence and ?lme4::troubleshooting.”


Generalized linear mixed model fit by maximum likelihood (Laplace
  Approximation) [glmerMod]
 Family: binomial  ( logit )
Formula: correct_prox_choice ~ condition * group + shape + oneStepAcc_c +  
    (1 + condition | subID)
   Data: prox_df
Control: 
glmerControl(optimizer = "Nelder_Mead", optCtrl = list(maxfun = 1e+05))
 Subset: closer_dist == 2

      AIC       BIC    logLik -2*log(L)  df.resid 
   7862.4    7924.5   -3922.2    7844.4      7271 

Scaled residuals: 
    Min      1Q  Median      3Q     Max 
-3.8538 -0.9488  0.4027  0.6238  1.3224 

Random effects:
 Groups Name        Variance Std.Dev. Corr  
 subID  (Intercept) 0.5200   0.7211         
        condition2  0.4593   0.6777   -0.67 
Number of obs: 7280, groups:  subID, 182

Fixed effects:
                  Estimate Std. Error z value Pr(>|z|)    
(Intercept)        1.53121    0.11240  13.623  < 2e-16 ***
condition2        -0.55953    0.10927  -5.120 3.05e-07 ***
group2            -0.13111    0.13958  -0.939    0.348   

,2.5 %,97.5 %
(Intercept),1.3109152,1.7514990
condition2,-0.7737015,-0.3453599
group2,-0.4046827,0.1424654
shape2,-0.2564548,0.1617034
oneStepAcc_c,2.3950640,3.3529675
condition2:group2,-0.2314856,0.3737302


In [40]:
model_2v1_diag_prox <- glmer(
  correct_prox_choice ~ condition * group + shape + oneStepAcc_c +
    (1 + condition | subID),
  data = prox_df,
  subset = closer_dist==2.5,
  family = binomial(link = "logit"),
  control = glmerControl(
    optimizer = "Nelder_Mead",
    optCtrl = list(maxfun = 100000)
  )
)

summary(model_2v1_diag_prox)
confint(model_2v1_diag_prox, parm = "beta_", method = "Wald")

Generalized linear mixed model fit by maximum likelihood (Laplace
  Approximation) [glmerMod]
 Family: binomial  ( logit )
Formula: correct_prox_choice ~ condition * group + shape + oneStepAcc_c +  
    (1 + condition | subID)
   Data: prox_df
Control: 
glmerControl(optimizer = "Nelder_Mead", optCtrl = list(maxfun = 1e+05))
 Subset: closer_dist == 2.5

      AIC       BIC    logLik -2*log(L)  df.resid 
   7146.1    7207.2   -3564.1    7128.1      6543 

Scaled residuals: 
    Min      1Q  Median      3Q     Max 
-3.4628 -0.9438  0.4409  0.6167  1.2726 

Random effects:
 Groups Name        Variance Std.Dev. Corr  
 subID  (Intercept) 0.3303   0.5747         
        condition2  0.3497   0.5914   -0.59 
Number of obs: 6552, groups:  subID, 182

Fixed effects:
                  Estimate Std. Error z value Pr(>|z|)    
(Intercept)        1.26444    0.09990  12.657  < 2e-16 ***
condition2        -0.41350    0.10496  -3.940 8.16e-05 ***
group2            -0.21454    0.12316  -1.742  0.08152 

,2.5 %,97.5 %
(Intercept),1.06863266,1.46024720
condition2,-0.61921079,-0.20778784
group2,-0.45593641,0.02685309
shape2,0.10651083,0.49633223
oneStepAcc_c,2.23681262,3.10353980
condition2:group2,-0.04358445,0.54047342


In [47]:
model_3v2_diag_prox <- glmer(
  correct_prox_choice ~ condition * group + shape + oneStepAcc_c +
    (1 + condition | subID),
  data = prox_df,
  subset = closer_dist==3,
  family = binomial(link = "logit"),
  control = glmerControl(
    optimizer = "Nelder_Mead",
    optCtrl = list(maxfun = 100000)
  )
)

summary(model_3v2_diag_prox)
confint(model_3v2_diag_prox, parm = "beta_", method = "Wald")

Generalized linear mixed model fit by maximum likelihood (Laplace
  Approximation) [glmerMod]
 Family: binomial  ( logit )
Formula: correct_prox_choice ~ condition * group + shape + oneStepAcc_c +  
    (1 + condition | subID)
   Data: prox_df
Control: 
glmerControl(optimizer = "Nelder_Mead", optCtrl = list(maxfun = 1e+05))
 Subset: closer_dist == 3

      AIC       BIC    logLik -2*log(L)  df.resid 
   8574.7    8636.8   -4278.4    8556.7      7271 

Scaled residuals: 
    Min      1Q  Median      3Q     Max 
-3.4647 -1.0150  0.4523  0.7333  1.5461 

Random effects:
 Groups Name        Variance Std.Dev. Corr  
 subID  (Intercept) 0.4183   0.6467         
        condition2  0.2318   0.4814   -0.35 
Number of obs: 7280, groups:  subID, 182

Fixed effects:
                  Estimate Std. Error z value Pr(>|z|)    
(Intercept)        1.21751    0.10454  11.646  < 2e-16 ***
condition2        -0.34586    0.09224  -3.750 0.000177 ***
group2            -0.27373    0.12491  -2.191 0.028421 * 

,2.5 %,97.5 %
(Intercept),1.0126163,1.42240081
condition2,-0.5266371,-0.16507521
group2,-0.5185403,-0.02891093
shape2,-0.3651345,0.07757839
oneStepAcc_c,1.8758641,2.87601740
condition2:group2,-0.1005696,0.40754896


In [48]:
model_2v2_diag_prox <- glmer(
  correct_prox_choice ~ condition * group + shape + oneStepAcc_c +
    (1 + condition | subID),
  data = prox_df,
  subset = closer_dist==2.2,
  family = binomial(link = "logit"),
  control = glmerControl(
    optimizer = "Nelder_Mead",
    optCtrl = list(maxfun = 100000)
  )
)

summary(model_2v2_diag_prox)
confint(model_2v2_diag_prox, parm = "beta_", method = "Wald")

Generalized linear mixed model fit by maximum likelihood (Laplace
  Approximation) [glmerMod]
 Family: binomial  ( logit )
Formula: correct_prox_choice ~ condition * group + shape + oneStepAcc_c +  
    (1 + condition | subID)
   Data: prox_df
Control: 
glmerControl(optimizer = "Nelder_Mead", optCtrl = list(maxfun = 1e+05))
 Subset: closer_dist == 2.2

      AIC       BIC    logLik -2*log(L)  df.resid 
  10589.8   10652.7   -5285.9   10571.8      7999 

Scaled residuals: 
    Min      1Q  Median      3Q     Max 
-1.5783 -0.9266 -0.3667  0.9209  2.8759 

Random effects:
 Groups Name        Variance Std.Dev. Corr  
 subID  (Intercept) 0.4121   0.6420         
        condition2  0.1729   0.4158   -0.49 
Number of obs: 8008, groups:  subID, 182

Fixed effects:
                  Estimate Std. Error z value Pr(>|z|)    
(Intercept)       -0.52862    0.09537  -5.543 2.98e-08 ***
condition2         0.03606    0.07962   0.453  0.65065    
group2             0.36019    0.11746   3.067  0.00216 

,2.5 %,97.5 %
(Intercept),-0.7155405,-0.3416992
condition2,-0.1200014,0.1921188
group2,0.1299844,0.5904033
shape2,0.1877382,0.5895534
oneStepAcc_c,-1.5837517,-0.6777485
condition2:group2,-0.2573092,0.1865680


Arena reconstruction accuracy: note that this is not a single-trial model as there is only one accuracy value (Fisher z-scored correlation between actual grid and reconstructed grid). I have therefore continued to use the equivalent of the python version previously specified.

In [60]:
grids_df <- read.csv('/Users/leonieglitz/Desktop/Grids_project/data/grids_stacked.csv')

grids_df$condition <- factor(grids_df$condition)
grids_df$group <- factor(grids_df$group)
grids_df$sub    <- factor(grids_df$sub)



model <- lmer(
  arena ~ condition * group + oneStep +
    (1 | sub),
  data = grids_df
)

summary(model)

Linear mixed model fit by REML ['lmerMod']
Formula: arena ~ condition * group + oneStep + (1 | sub)
   Data: grids_df

REML criterion at convergence: -14.1

Scaled residuals: 
    Min      1Q  Median      3Q     Max 
-3.8113 -0.1573  0.0727  0.3675  3.0849 

Random effects:
 Groups   Name        Variance Std.Dev.
 sub      (Intercept) 0.05458  0.2336  
 Residual             0.01885  0.1373  
Number of obs: 184, groups:  sub, 92

Fixed effects:
                   Estimate Std. Error t value
(Intercept)        0.306909   0.074980   4.093
condition1         0.015754   0.029619   0.532
group1            -0.014287   0.056696  -0.252
oneStep            0.603254   0.080790   7.467
condition1:group1 -0.005467   0.040789  -0.134

Correlation of Fixed Effects:
            (Intr) cndtn1 group1 oneStp
condition1  -0.176                     
group1      -0.444  0.260              
oneStep     -0.834 -0.025  0.051       
cndtn1:grp1  0.229 -0.723 -0.361 -0.103